In [1]:
import pandas as pd
import re

df = pd.read_csv('radvlm_results.csv')
df.head()

,image_path,ground_truth,response
0,/home/ai-user/disk/bharatradio/bharatradio_all...,pleuralthickening,The following abnormalities are present: lung ...
1,/home/ai-user/disk/bharatradio/bharatradio_all...,pleuralthickening,The following abnormalities are present: cardi...
2,/home/ai-user/disk/bharatradio/bharatradio_all...,pleuralthickening,The following abnormalities are present: atele...
3,/home/ai-user/disk/bharatradio/bharatradio_all...,pleuralthickening,The following abnormalities are identified: lu...
4,/home/ai-user/disk/bharatradio/bharatradio_all...,pleuralthickening,The Chest X-ray appears to be clear of any abn...


In [2]:
chexpert_to_5c = {
    "atelectasis": None,
    "cardiomegaly": None,
    "consolidation": "consolidation",
    "edema": None,
    "enlarged cardio": None,
    "fracture": None,
    "lung lesion": None,
    "lung opacity": None,
    "no finding": "normal",
    "pleural effusion": "pleuraleffusion",
    "pleural other": ["pleuralcalcification", "pleuralthickening"],
    "pneumonia": None,
    "pneumothorax": "pneumothorax",
    "support devices": None
}

In [3]:
# Print all the unique values in 'ground_truth' column
print(df['ground_truth'].unique())


['pleuralthickening' 'bvm' 'pneumothorax' 'hilarlymphadenopathy'
 'bronchiectasis' 'normal' 'consolidation' 'tenting' 'oldtb' 'cavity'
 'fibrosis' 'pleuralcalcification' 'lunglobecollapse' 'cannonball'
 'trachealshift' 'tuberculosis' 'milliarytb' 'pleuraleffusion'
 'mediastinalshift' 'nodules']


In [4]:
unique_pathologies = df['ground_truth'].unique()

In [5]:
sorted(unique_pathologies)

['bronchiectasis',
 'bvm',
 'cannonball',
 'cavity',
 'consolidation',
 'fibrosis',
 'hilarlymphadenopathy',
 'lunglobecollapse',
 'mediastinalshift',
 'milliarytb',
 'nodules',
 'normal',
 'oldtb',
 'pleuralcalcification',
 'pleuraleffusion',
 'pleuralthickening',
 'pneumothorax',
 'tenting',
 'trachealshift',
 'tuberculosis']

In [6]:
def extract_pathologies(text):
    text = text.lower()
    if "clear of any abnormalities" in text or "no abnormalities" in text:
        return "no finding"
    matches = re.findall(r"\b(?:atelectasis|cardiomegaly|consolidation|edema|enlarged cardio|fracture|lung lesion|lung opacity|pleural effusion|pleural other|pneumonia|pneumothorax|support devices)\b", text)
    return ", ".join(matches) if matches else "no finding"

df['pathologies_radvlm'] = df['response'].apply(extract_pathologies)

In [7]:
df_normal_abnormal = df.copy()

def classify_normal_abnormal(row):
    gt = row['ground_truth'].strip().lower()
    pred = row['pathologies_radvlm'].strip().lower()

    return pd.Series({
        'TP': int(gt != 'normal' and pred != 'no finding'),
        'TN': int(gt == 'normal' and pred == 'no finding'),
        'FP': int(gt == 'normal' and pred != 'no finding'),
        'FN': int(gt != 'normal' and pred == 'no finding')
    })

df_normal_abnormal[['TP', 'TN', 'FP', 'FN']] = df_normal_abnormal.apply(classify_normal_abnormal, axis=1)

df_normal_abnormal.to_csv("normal-abnormal.csv", index=False)

In [8]:
df_pathology = df.copy()

# Invert chexpert_to_5c for reverse mapping
from collections import defaultdict

reverse_mapping = defaultdict(list)
for chexpert, mapped in chexpert_to_5c.items():
    if mapped is None:
        continue
    if isinstance(mapped, list):
        for val in mapped:
            reverse_mapping[chexpert].append(val)
    else:
        reverse_mapping[chexpert].append(mapped)

valid_5c_classes = set()
for v in reverse_mapping.values():
    valid_5c_classes.update(v)

def classify_pathology(row):
    gt = row['ground_truth'].strip().lower()
    preds = [p.strip() for p in row['pathologies_radvlm'].strip().lower().split(',')]
    result = {'TP': 0, 'TN': 0, 'FP': 0, 'FN': 0, 'different_pathology': 0}

    # Normal/Abnormal logic
    if gt == 'normal' and 'no finding' in preds:
        result['TN'] = 1
    elif gt != 'normal' and 'no finding' in preds:
        result['FN'] = 1
    elif gt == 'normal' and 'no finding' not in preds:
        result['FP'] = 1
    else:
        # Check if any predicted chexpert label maps to the GT label
        gt_matched = False
        for pred in preds:
            mapped_5c = reverse_mapping.get(pred, [])
            if gt in mapped_5c:
                gt_matched = True
                break
        if gt_matched:
            result['TP'] = 1
        else:
            result['FN'] = 1

    return pd.Series(result)


df_pathology[['TP', 'TN', 'FP', 'FN', 'different_pathology']] = df_pathology.apply(classify_pathology, axis=1)

# Remove rows where radvlm predictions have no 5C class mapping
valid_5c_classes = {'consolidation', 'pleuraleffusion', 'pleuralcalcification', 'pleuralthickening', 'pneumothorax', 'normal'}
df_pathology = df_pathology[df_pathology['ground_truth'].isin(valid_5c_classes)]

df_pathology.to_csv("pathology_wise_classification.csv", index=False)

# Consider this


In [18]:
import pandas as pd
import re

df = pd.read_csv('radvlm_results.csv')
df.head()

chexpert_to_5c = {
    "atelectasis": None,
    "cardiomegaly": None,
    "consolidation": "consolidation",
    "edema": None,
    "enlarged cardio": None,
    "fracture": None,
    "lung lesion": None,
    "lung opacity": None,
    "no finding": "normal",
    "pleural effusion": "pleuraleffusion",
    "pleural other": ["pleuralcalcification", "pleuralthickening"],
    "pneumonia": None,
    "pneumothorax": "pneumothorax",
    "support devices": None
}

def extract_pathologies(text):
    text = text.lower()
    if "clear of any abnormalities" in text or "no abnormalities" in text:
        return "no finding"
    matches = re.findall(r"\b(?:atelectasis|cardiomegaly|consolidation|edema|enlarged cardio|fracture|lung lesion|lung opacity|pleural effusion|pleural other|pneumonia|pneumothorax|support devices)\b", text)
    return ", ".join(matches) if matches else "no finding"

df['pathologies_radvlm'] = df['response'].apply(extract_pathologies)

# Normal/Abnormal classification (unchanged)
df_normal_abnormal = df.copy()

def classify_normal_abnormal(row):
    gt = row['ground_truth'].strip().lower()
    pred = row['pathologies_radvlm'].strip().lower()
    return pd.Series({
        'TP': int(gt != 'normal' and pred != 'no finding'),
        'TN': int(gt == 'normal' and pred == 'no finding'),
        'FP': int(gt == 'normal' and pred != 'no finding'),
        'FN': int(gt != 'normal' and pred == 'no finding')
    })

df_normal_abnormal[['TP', 'TN', 'FP', 'FN']] = df_normal_abnormal.apply(classify_normal_abnormal, axis=1)
df_normal_abnormal.to_csv("normal-abnormal.csv", index=False)

# Pathology-specific classification (modified)
df_pathology = df.copy()

# Create mapping from chexpert labels to 5C classes
from collections import defaultdict
chexpert_to_5c_mapping = {}
for chexpert, mapped in chexpert_to_5c.items():
    if mapped is None:
        continue
    if isinstance(mapped, list):
        # Store the list as is, don't just take first value
        chexpert_to_5c_mapping[chexpert] = mapped
    else:
        chexpert_to_5c_mapping[chexpert] = [mapped]  # Convert single values to list for consistency

# Define the valid 5C classes we're interested in
target_pathologies = ['consolidation', 'pleuraleffusion', 'pleuralcalcification', 'pleuralthickening', 'pneumothorax']

def classify_pathology_specific(row):
    gt = row['ground_truth'].strip().lower()
    preds_text = row['pathologies_radvlm'].strip().lower()
    
    # Extract predicted chexpert labels
    pred_chexpert_labels = []
    if preds_text != 'no finding':
        pred_chexpert_labels = [p.strip() for p in preds_text.split(',')]
    
    # Convert predicted chexpert labels to 5C classes
    pred_5c_classes = []
    for pred_chexpert in pred_chexpert_labels:
        if pred_chexpert in chexpert_to_5c_mapping:
            mapped_5c_list = chexpert_to_5c_mapping[pred_chexpert]
            for mapped_5c in mapped_5c_list:
                if mapped_5c not in pred_5c_classes:
                    pred_5c_classes.append(mapped_5c)
    
    result = {}
    
    # For each target pathology, calculate TP, TN, FP, FN
    for pathology in target_pathologies:
        pathology_in_gt = (gt == pathology)
        pathology_in_pred = (pathology in pred_5c_classes)
        
        if pathology_in_gt and pathology_in_pred:
            # Ground truth is this pathology AND prediction contains this pathology
            result[f'{pathology}_TP'] = 1
            result[f'{pathology}_TN'] = 0
            result[f'{pathology}_FP'] = 0
            result[f'{pathology}_FN'] = 0
        elif pathology_in_gt and not pathology_in_pred:
            # Ground truth is this pathology BUT prediction does NOT contain this pathology (False Negative)
            result[f'{pathology}_TP'] = 0
            result[f'{pathology}_TN'] = 0
            result[f'{pathology}_FP'] = 0
            result[f'{pathology}_FN'] = 1
        elif not pathology_in_gt and pathology_in_pred:
            # Ground truth is NOT this pathology BUT prediction contains this pathology (False Positive)
            result[f'{pathology}_TP'] = 0
            result[f'{pathology}_TN'] = 0
            result[f'{pathology}_FP'] = 1
            result[f'{pathology}_FN'] = 0
        else:
            # Ground truth is NOT this pathology AND prediction does NOT contain this pathology (True Negative)
            result[f'{pathology}_TP'] = 0
            result[f'{pathology}_TN'] = 1
            result[f'{pathology}_FP'] = 0
            result[f'{pathology}_FN'] = 0
    
    # Handle normal case separately
    normal_in_gt = (gt == 'normal')
    normal_in_pred = (len(pred_5c_classes) == 0 or 'normal' in pred_5c_classes)
    
    if normal_in_gt and normal_in_pred:
        result['normal_TP'] = 1
        result['normal_TN'] = 0
        result['normal_FP'] = 0
        result['normal_FN'] = 0
    elif normal_in_gt and not normal_in_pred:
        result['normal_TP'] = 0
        result['normal_TN'] = 0
        result['normal_FP'] = 0
        result['normal_FN'] = 1
    elif not normal_in_gt and normal_in_pred:
        result['normal_TP'] = 0
        result['normal_TN'] = 0
        result['normal_FP'] = 1
        result['normal_FN'] = 0
    else:
        result['normal_TP'] = 0
        result['normal_TN'] = 1
        result['normal_FP'] = 0
        result['normal_FN'] = 0
    
    return pd.Series(result)

# Apply the classification
classification_results = df_pathology.apply(classify_pathology_specific, axis=1)
df_pathology = pd.concat([df_pathology, classification_results], axis=1)

# Filter to only include rows with valid 5C classes
valid_5c_classes = {'consolidation', 'pleuraleffusion', 'pleuralcalcification', 'pleuralthickening', 'pneumothorax', 'normal'}
df_pathology = df_pathology[df_pathology['ground_truth'].isin(valid_5c_classes)]

df_pathology.to_csv("pathology_wise_classification_claude2.csv", index=False)

# Debug: Check the mapping for pleural other
print("Debug: Chexpert to 5C mapping:")
for chexpert, mapped in chexpert_to_5c_mapping.items():
    print(f"  {chexpert} -> {mapped}")

print(f"\nDebug: Sample predictions containing 'pleural other':")
sample_pleural = df_pathology[df_pathology['pathologies_radvlm'].str.contains('pleural other', na=False)].head(3)
for idx, row in sample_pleural.iterrows():
    print(f"  Row {idx}: GT='{row['ground_truth']}', Pred='{row['pathologies_radvlm']}'")

# Print summary statistics for each pathology
print("Summary Statistics for Each Pathology:")
print("=" * 50)

all_pathologies = target_pathologies + ['normal']
for pathology in all_pathologies:
    tp = df_pathology[f'{pathology}_TP'].sum()
    tn = df_pathology[f'{pathology}_TN'].sum()
    fp = df_pathology[f'{pathology}_FP'].sum()
    fn = df_pathology[f'{pathology}_FN'].sum()
    
    print(f"\n{pathology.upper()}:")
    print(f"  True Positives:  {tp}")
    print(f"  True Negatives:  {tn}")
    print(f"  False Positives: {fp}")
    print(f"  False Negatives: {fn}")
    
    # Calculate metrics
    if tp + fp > 0:
        precision = tp / (tp + fp)
    else:
        precision = 0
    
    if tp + fn > 0:
        recall = tp / (tp + fn)
    else:
        recall = 0
    
    if tp + tn + fp + fn > 0:
        accuracy = (tp + tn) / (tp + tn + fp + fn)
    else:
        accuracy = 0
    
    if precision + recall > 0:
        f1 = 2 * (precision * recall) / (precision + recall)
    else:
        f1 = 0
    
    print(f"  Precision: {precision:.3f}")
    print(f"  Recall:    {recall:.3f}")
    print(f"  F1-Score:  {f1:.3f}")
    print(f"  Accuracy:  {accuracy:.3f}")

Debug: Chexpert to 5C mapping:
  consolidation -> ['consolidation']
  no finding -> ['normal']
  pleural effusion -> ['pleuraleffusion']
  pleural other -> ['pleuralcalcification', 'pleuralthickening']
  pneumothorax -> ['pneumothorax']

Debug: Sample predictions containing 'pleural other':
  Row 27: GT='pleuralthickening', Pred='lung opacity, pleural other'
  Row 29: GT='pleuralthickening', Pred='lung opacity, pleural other'
  Row 35: GT='pleuralthickening', Pred='lung opacity, pleural other'
Summary Statistics for Each Pathology:

CONSOLIDATION:
  True Positives:  18
  True Negatives:  445
  False Positives: 55
  False Negatives: 82
  Precision: 0.247
  Recall:    0.180
  F1-Score:  0.208
  Accuracy:  0.772

PLEURALEFFUSION:
  True Positives:  76
  True Negatives:  362
  False Positives: 138
  False Negatives: 24
  Precision: 0.355
  Recall:    0.760
  F1-Score:  0.484
  Accuracy:  0.730

PLEURALCALCIFICATION:
  True Positives:  12
  True Negatives:  493
  False Positives: 7
  False 